In [1]:

import sys
import os
import polars as pl
from pathlib import Path

In [2]:
processed_path = Path('..') / '02_data' / 'processed'
df = pl.read_parquet(processed_path / 'ecommerce_long_format.parquet')

print("Daten erfolgreich geladen!")

Daten erfolgreich geladen!


## Lag Features

In [3]:
sys.path.append(os.path.abspath("../03_src"))
from feature import add_lag


df_lagged = add_lag(df,[3, 6, 12])
df_lagged

datetime,platform,sales,lag_3,lag_6,lag_12
date,str,f64,f64,f64,f64
2010-01-31,"""amazon""",162.50523,null,null,null
2010-02-28,"""amazon""",131.234988,null,null,null
2010-03-31,"""amazon""",142.409296,null,null,null
2010-04-30,"""amazon""",265.024017,162.50523,null,null
2010-05-31,"""amazon""",154.458817,131.234988,null,null
…,…,…,…,…,…
2019-08-31,"""trendyol""",82.833966,62.742393,83.022732,69.299408
2019-09-30,"""trendyol""",79.782013,113.855833,85.74682,81.932279
2019-10-31,"""trendyol""",77.831401,95.377609,72.848862,51.240824


## Rolling Window

In [4]:
from feature import add_rolling_mean

df_rolling = add_rolling_mean(df_lagged, [3, 6, 12])
df_rolling

datetime,platform,sales,lag_3,lag_6,lag_12,rolling_mean_3_months,rolling_mean_6_months,rolling_mean_12_months
date,str,f64,f64,f64,f64,f64,f64,f64
2010-01-31,"""amazon""",162.50523,null,null,null,null,null,null
2010-02-28,"""amazon""",131.234988,null,null,null,null,null,null
2010-03-31,"""amazon""",142.409296,null,null,null,145.383171,null,null
2010-04-30,"""amazon""",265.024017,162.50523,null,null,179.5561,null,null
2010-05-31,"""amazon""",154.458817,131.234988,null,null,187.297376,null,null
…,…,…,…,…,…,…,…,…
2019-08-31,"""trendyol""",82.833966,62.742393,83.022732,69.299408,97.355802,85.56758,77.914245
2019-09-30,"""trendyol""",79.782013,113.855833,85.74682,81.932279,85.997863,84.573446,77.735056
2019-10-31,"""trendyol""",77.831401,95.377609,72.848862,51.240824,80.149127,85.403869,79.950937


## Time based features

In [5]:
from feature import add_weekend_features

df_time_based = add_weekend_features(df_rolling)
df_time_based

datetime,platform,sales,lag_3,lag_6,lag_12,rolling_mean_3_months,rolling_mean_6_months,rolling_mean_12_months,day_name,is_weekend
date,str,f64,f64,f64,f64,f64,f64,f64,str,i8
2010-01-31,"""amazon""",162.50523,null,null,null,null,null,null,"""Sunday""",1
2010-02-28,"""amazon""",131.234988,null,null,null,null,null,null,"""Sunday""",1
2010-03-31,"""amazon""",142.409296,null,null,null,145.383171,null,null,"""Wednesday""",0
2010-04-30,"""amazon""",265.024017,162.50523,null,null,179.5561,null,null,"""Friday""",0
2010-05-31,"""amazon""",154.458817,131.234988,null,null,187.297376,null,null,"""Monday""",0
…,…,…,…,…,…,…,…,…,…,…
2019-08-31,"""trendyol""",82.833966,62.742393,83.022732,69.299408,97.355802,85.56758,77.914245,"""Saturday""",1
2019-09-30,"""trendyol""",79.782013,113.855833,85.74682,81.932279,85.997863,84.573446,77.735056,"""Monday""",0
2019-10-31,"""trendyol""",77.831401,95.377609,72.848862,51.240824,80.149127,85.403869,79.950937,"""Thursday""",0


## Momentum Features

In [6]:
from feature import add_momentum_features

df_momentum = add_momentum_features(df_time_based)
df_momentum

datetime,platform,sales,lag_3,lag_6,lag_12,rolling_mean_3_months,rolling_mean_6_months,rolling_mean_12_months,day_name,is_weekend,mom_growth,yoy_growth
date,str,f64,f64,f64,f64,f64,f64,f64,str,i8,f64,f64
2010-01-31,"""amazon""",162.50523,null,null,null,null,null,null,"""Sunday""",1,null,null
2010-02-28,"""amazon""",131.234988,null,null,null,null,null,null,"""Sunday""",1,-0.192426,null
2010-03-31,"""amazon""",142.409296,null,null,null,145.383171,null,null,"""Wednesday""",0,0.085147,null
2010-04-30,"""amazon""",265.024017,162.50523,null,null,179.5561,null,null,"""Friday""",0,0.861002,null
2010-05-31,"""amazon""",154.458817,131.234988,null,null,187.297376,null,null,"""Monday""",0,-0.417189,null
…,…,…,…,…,…,…,…,…,…,…,…,…
2019-08-31,"""trendyol""",82.833966,62.742393,83.022732,69.299408,97.355802,85.56758,77.914245,"""Saturday""",1,-0.131516,0.195306
2019-09-30,"""trendyol""",79.782013,113.855833,85.74682,81.932279,85.997863,84.573446,77.735056,"""Monday""",0,-0.036844,-0.026244
2019-10-31,"""trendyol""",77.831401,95.377609,72.848862,51.240824,80.149127,85.403869,79.950937,"""Thursday""",0,-0.024449,0.518933


## Zyklische Features

In [7]:
from feature import add_cyclical_month_features

df_zyclic = add_cyclical_month_features(df_momentum)
df_zyclic

datetime,platform,sales,lag_3,lag_6,lag_12,rolling_mean_3_months,rolling_mean_6_months,rolling_mean_12_months,day_name,is_weekend,mom_growth,yoy_growth,month,quarter,month_sin,month_cos
date,str,f64,f64,f64,f64,f64,f64,f64,str,i8,f64,f64,i8,i8,f64,f64
2010-01-31,"""amazon""",162.50523,null,null,null,null,null,null,"""Sunday""",1,null,null,1,1,0.5,0.866025
2010-02-28,"""amazon""",131.234988,null,null,null,null,null,null,"""Sunday""",1,-0.192426,null,2,1,0.866025,0.5
2010-03-31,"""amazon""",142.409296,null,null,null,145.383171,null,null,"""Wednesday""",0,0.085147,null,3,1,1.0,6.1232e-17
2010-04-30,"""amazon""",265.024017,162.50523,null,null,179.5561,null,null,"""Friday""",0,0.861002,null,4,2,0.866025,-0.5
2010-05-31,"""amazon""",154.458817,131.234988,null,null,187.297376,null,null,"""Monday""",0,-0.417189,null,5,2,0.5,-0.866025
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2019-08-31,"""trendyol""",82.833966,62.742393,83.022732,69.299408,97.355802,85.56758,77.914245,"""Saturday""",1,-0.131516,0.195306,8,3,-0.866025,-0.5
2019-09-30,"""trendyol""",79.782013,113.855833,85.74682,81.932279,85.997863,84.573446,77.735056,"""Monday""",0,-0.036844,-0.026244,9,3,-1.0,-1.8370e-16
2019-10-31,"""trendyol""",77.831401,95.377609,72.848862,51.240824,80.149127,85.403869,79.950937,"""Thursday""",0,-0.024449,0.518933,10,4,-0.866025,0.5


## Benchmarks

In [ ]:
from feature import add_historical_benchmarks

df_benchmarks = add_historical_benchmarks(df_zyclic)
df_benchmarks

datetime,platform,sales,lag_3,lag_6,lag_12,rolling_mean_3_months,rolling_mean_6_months,rolling_mean_12_months,day_name,is_weekend,mom_growth,yoy_growth,month,quarter,month_sin,month_cos,year,avg_sales_last_year_same_month
date,str,f64,f64,f64,f64,f64,f64,f64,str,i8,f64,f64,i8,i8,f64,f64,i32,f64
2010-01-31,"""amazon""",162.50523,null,null,null,null,null,null,"""Sunday""",1,null,null,1,1,0.5,0.866025,2010,null
2010-02-28,"""amazon""",131.234988,null,null,null,null,null,null,"""Sunday""",1,-0.192426,null,2,1,0.866025,0.5,2010,null
2010-03-31,"""amazon""",142.409296,null,null,null,145.383171,null,null,"""Wednesday""",0,0.085147,null,3,1,1.0,6.1232e-17,2010,null
2010-04-30,"""amazon""",265.024017,162.50523,null,null,179.5561,null,null,"""Friday""",0,0.861002,null,4,2,0.866025,-0.5,2010,null
2010-05-31,"""amazon""",154.458817,131.234988,null,null,187.297376,null,null,"""Monday""",0,-0.417189,null,5,2,0.5,-0.866025,2010,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2019-08-31,"""trendyol""",82.833966,62.742393,83.022732,69.299408,97.355802,85.56758,77.914245,"""Saturday""",1,-0.131516,0.195306,8,3,-0.866025,-0.5,2019,69.299408
2019-09-30,"""trendyol""",79.782013,113.855833,85.74682,81.932279,85.997863,84.573446,77.735056,"""Monday""",0,-0.036844,-0.026244,9,3,-1.0,-1.8370e-16,2019,81.932279
2019-10-31,"""trendyol""",77.831401,95.377609,72.848862,51.240824,80.149127,85.403869,79.950937,"""Thursday""",0,-0.024449,0.518933,10,4,-0.866025,0.5,2019,51.240824


## Ratio Features

In [10]:
from feature import add_ratio_features

df_ratio = add_ratio_features(df_benchmarks)
df_ratio

ImportError: cannot import name 'add_ratio_features' from 'feature' (c:\Users\maxkr\E-Commerce-Time-Series-Forecasting\03_src\feature.py)

## Difference Features

In [ ]:
from feature import add_difference_features

df_final = add_difference_features(df_ratio)
df_final

['datetime',
 'platform',
 'sales',
 'lag_3',
 'lag_6',
 'lag_12',
 'rolling_mean_3_months',
 'rolling_mean_6_months',
 'rolling_mean_12_months',
 'day_name',
 'is_weekend',
 'mom_growth',
 'yoy_growth',
 'month',
 'quarter',
 'month_sin',
 'month_cos',
 'year',
 'avg_sales_last_year_same_month']